# GeoSR-AI — Notebook 01: Data Pipeline & Preprocessing
**Deep Learning Based Super Resolution Mapping from Medium-Resolution Satellite Imagery**

### Overview
This notebook demonstrates:
1. Environment and remote sensing package inspection
2. YAML configuration loading
3. Multi-band geospatial raster loading and metadata preservation (`RasterLoader`, `GeoMetadata`)
4. Remote Sensing aware normalization (`SatelliteNormalizer`)
5. Cloud glint and invalid pixel mask extraction (`CloudMasker`)
6. Sensor point-spread-function (PSF) controlled synthetic degradation (`ControlledDegradation`)
7. Spatial patch extraction (`RasterTiler`)
8. PyTorch `PairedSatelliteDataset` & DataLoader setup with geographic scene-level partitioning

In [ ]:
import os
import sys
try:
    import yaml
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyyaml"])
    import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Add GeoSR-AI root to sys.path
sys.path.insert(0, os.path.abspath(".."))

from geospatial.raster_loader import RasterLoader
from geospatial.metadata import GeoMetadata
from preprocessing.normalization import SatelliteNormalizer
from preprocessing.cloud_mask import CloudMasker
from preprocessing.degradation import ControlledDegradation
from preprocessing.tiling import RasterTiler
from datasets.paired_dataset import PairedSatelliteDataset, create_dataloaders

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

## 1. Load Base Configuration (`configs/base.yaml`)

In [ ]:
config_path = "../configs/base.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
print("Base Experiment Configuration:")
print(yaml.dump(config, default_flow_style=False))

## 2. Test Geospatial Normalization & Controlled Degradation

In [ ]:
# Initialize normalizer and controlled degradation module
normalizer = SatelliteNormalizer(method="percentile", percentiles=(1.0, 99.0))
degradation = ControlledDegradation(scale_factor=4, blur_sigma=1.5, noise_type="gaussian", noise_sigma=0.005)

# Create synthetic land cover patch (Forest, Water, Agriculture, Urban)
ps = 128
hr_patch = np.zeros((3, ps, ps), dtype=np.float32)
hr_patch[0, :ps//2, :ps//2] = 35.0; hr_patch[1, :ps//2, :ps//2] = 135.0; hr_patch[2, :ps//2, :ps//2] = 35.0   # Forest
hr_patch[0, ps//2:, :ps//2] = 20.0; hr_patch[1, ps//2:, :ps//2] = 80.0; hr_patch[2, ps//2:, :ps//2] = 200.0  # Water
hr_patch[0, :ps//2, ps//2:] = 180.0; hr_patch[1, :ps//2, ps//2:] = 210.0; hr_patch[2, :ps//2, ps//2:] = 40.0 # Vegetation
hr_patch[0, ps//2:, ps//2:] = 160.0; hr_patch[1, ps//2:, ps//2:] = 160.0; hr_patch[2, ps//2:, ps//2:] = 160.0 # Urban

hr_norm, stats = normalizer.normalize(hr_patch)
lr_norm = degradation.degrade(hr_norm)

print(f"HR Reference shape: {hr_norm.shape}, range: [{hr_norm.min():.3f}, {hr_norm.max():.3f}]")
print(f"LR Input shape:     {lr_norm.shape}, range: [{lr_norm.min():.3f}, {lr_norm.max():.3f}]")

## 3. Visualize HR Reference vs Controlled Synthetic LR Pair

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(np.transpose(hr_norm, (1, 2, 0)))
axes[0].set_title(f"HR Reference Target ({ps}x{ps})")
axes[0].axis("off")

axes[1].imshow(np.transpose(lr_norm, (1, 2, 0)))
axes[1].set_title(f"Synthetic LR Input ({lr_norm.shape[1]}x{lr_norm.shape[2]})")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. PyTorch Dataset & DataLoader Batching Verification

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    data_dir="../data/processed",
    batch_size=4,
    patch_size=128,
    scale_factor=4,
    num_workers=0
)

batch = next(iter(train_loader))
print(f"Train Batch LR Tensor shape: {batch['lr'].shape}")
print(f"Train Batch HR Tensor shape: {batch['hr'].shape}")
print(f"Scene IDs in batch:          {batch['scene_id']}")